# 🏇 NAR 全レース取得 v2 (94カラム完全対応)

## 主な特徴
- **94カラム完全対応**: 過去成績（5走分）と血統情報を含む完全なデータ取得
- **欠損チェック機能**: 全カラムをチェックし、不完全なレースのみ再取得
- **メモリ効率化**: 定期的なガベージコレクションでColab環境に最適化
- **カラムずれ防止**: 厳密な94カラム定義で整合性を保証

## 使い方
1. Google Driveをマウント
2. 設定（年度、月、保存先）を変更
3. 実行ブロックを実行

In [ ]:
# Google Driveをマウントする場合のみ実行してください
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================================
# 設定（ここを変更してください）
# ========================================
YEAR = 2025          # 対象年度
START_MONTH = 1      # 開始月 (1-12)
END_MONTH = 12       # 終了月 (1-12)
SAVE_DIR = '/content/drive/MyDrive/dai-keiba/data/raw' # 保存先フォルダ

In [ ]:
import requestsfrom bs4 import BeautifulSoupimport pandas as pdimport ioimport refrom datetime import datetime, date, timedeltaimport timeimport randomfrom tqdm.auto import tqdmimport gc# 82カラムの厳密な定義（データベースと完全一致）EXPECTED_COLUMNS = [    "日付","会場","レース番号","レース名","重賞","コースタイプ","距離","回り","天候","馬場状態",    "着順","枠","馬番","馬名","性齢","斤量","騎手","タイム","着差","人気","単勝オッズ","後3F",    "厩舎","馬体重(増減)","race_id","horse_id",    "past_1_date","past_1_rank","past_1_time","past_1_run_style","past_1_race_name","past_1_last_3f",    "past_1_horse_weight","past_1_jockey","past_1_condition","past_1_odds","past_1_weather",    "past_1_distance","past_1_course_type",    "past_2_date","past_2_rank","past_2_time","past_2_run_style","past_2_race_name","past_2_last_3f",    "past_2_horse_weight","past_2_jockey","past_2_condition","past_2_odds","past_2_weather",    "past_2_distance","past_2_course_type",    "past_3_date","past_3_rank","past_3_time","past_3_run_style","past_3_race_name","past_3_last_3f",    "past_3_horse_weight","past_3_jockey","past_3_condition","past_3_odds","past_3_weather",    "past_3_distance","past_3_course_type",    "past_4_date","past_4_rank","past_4_time","past_4_run_style","past_4_race_name","past_4_last_3f",    "past_4_horse_weight","past_4_jockey","past_4_condition","past_4_odds","past_4_weather",    "past_4_distance","past_4_course_type",    "past_5_date","past_5_rank","past_5_time","past_5_run_style","past_5_race_name","past_5_last_3f",    "past_5_horse_weight","past_5_jockey","past_5_condition","past_5_odds","past_5_weather",    "past_5_distance","past_5_course_type",    "father","mother","bms"]class RaceScraper:    def __init__(self):        self.headers = {            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"        }    def _get_soup(self, url, max_retries=3):        for attempt in range(max_retries):            try:                time.sleep(random.uniform(0.5, 1.0))                response = requests.get(url, headers=self.headers, timeout=15)                response.encoding = response.apparent_encoding                if response.status_code == 200:                    return BeautifulSoup(response.text, 'html.parser')                elif attempt < max_retries - 1:                    time.sleep(2 ** attempt)            except Exception as e:                if attempt < max_retries - 1:                    time.sleep(2 ** attempt)        return None    def get_past_races(self, horse_id, current_date, n_samples=5):        url = f"https://db.netkeiba.com/horse/result/{horse_id}/"        soup = self._get_soup(url)        if not soup:            return pd.DataFrame()        table = soup.select_one("table.db_h_race_results")        if not table:            tables = soup.find_all("table")            for t in tables:                if "着順" in t.text:                    table = t                    break        if not table:            return pd.DataFrame()        try:            df = pd.read_html(io.StringIO(str(table)))[0]            df = df.dropna(how='all')            df.columns = df.columns.astype(str).str.replace(r'\s+', '', regex=True)            if '日付' in df.columns:                df['date_obj'] = pd.to_datetime(df['日付'], format='%Y/%m/%d', errors='coerce')                df = df.dropna(subset=['date_obj'])                # 現在のレース日付より前のレースのみ                if current_date:                    df = df[df['date_obj'] < pd.to_datetime(current_date, format='%Y年%m月%d日', errors='coerce')]                df = df.sort_values('date_obj', ascending=False)            if n_samples:                df = df.head(n_samples)            if '通過' in df.columns:                df['run_style_val'] = df['通過'].apply(self.extract_run_style)            else:                df['run_style_val'] = ""            column_map = {                '日付': 'date', '天気': 'weather', 'レース名': 'race_name',                '着順': 'rank', '騎手': 'jockey', '馬場': 'condition',                'タイム': 'time', '上り': 'last_3f', '馬体重': 'horse_weight',                'run_style_val': 'run_style', '単勝': 'odds', 'オッズ': 'odds',                '距離': 'raw_distance'            }            df.rename(columns=column_map, inplace=True)            if 'raw_distance' in df.columns:                parsed = df['raw_distance'].apply(self._parse_dist)                df['course_type'] = parsed.apply(lambda x: x[0] if x else "")                df['distance'] = parsed.apply(lambda x: x[1] if x else "")            else:                df['course_type'] = ""                df['distance'] = ""            # データクリーニング            for col in ['rank', 'time', 'run_style', 'race_name', 'last_3f', 'horse_weight',                        'jockey', 'condition', 'odds', 'weather', 'distance', 'course_type', 'date']:                if col in df.columns:                    df[col] = df[col].fillna("").astype(str).str.replace('nan', '').str.strip()            return df        except Exception as e:            return pd.DataFrame()    def _parse_dist(self, x):        if not isinstance(x, str):            return ("", "")        surf = ""        dist = ""        if '芝' in x: surf = '芝'        elif 'ダ' in x: surf = 'ダート'        elif '障' in x: surf = '障害'        match = re.search(r'(\d+)', x)        if match:            dist = match.group(1)        return (surf, dist)    def extract_run_style(self, passing_str):        if not isinstance(passing_str, str):            return ""        try:            cleaned = re.sub(r'[^0-9-]', '', passing_str)            parts = [int(p) for p in cleaned.split('-') if p]            if not parts:                return ""            first_corner = parts[0]            if first_corner == 1: return "1"            elif first_corner <= 4: return "2"            elif first_corner <= 9: return "3"            else: return "4"        except:            return ""    def get_horse_profile(self, horse_id):        url = f"https://db.netkeiba.com/horse/ped/{horse_id}/"        soup = self._get_soup(url)        if not soup:            return {"father": "", "mother": "", "bms": ""}        data = {"father": "", "mother": "", "bms": ""}        try:            table = soup.select_one("table.blood_table")            if table:                rows = table.find_all("tr")                if len(rows) >= 17:                    r0 = rows[0].find_all("td")                    if r0:                        data["father"] = r0[0].text.strip().split('\n')[0].strip()                    r16 = rows[16].find_all("td")                    if len(r16) >= 2:                        data["mother"] = r16[0].text.strip().split('\n')[0].strip()                        data["bms"] = r16[1].text.strip().split('\n')[0].strip()        except:            pass        return datadef scrape_nar_race(url, existing_race_ids=None, max_retries=3):    """    Scrapes a single NAR race page from Netkeiba with full 82-column support.    Returns a pandas DataFrame with past performance and pedigree data.    """    print(f"Accessing URL: {url}...")    headers = {        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"    }    scraper = RaceScraper()    for attempt in range(max_retries):        try:            response = requests.get(url, headers=headers, timeout=15)            response.encoding = 'EUC-JP'            if response.status_code != 200:                if attempt < max_retries - 1:                    wait_time = 2 ** attempt                    print(f"Status {response.status_code}, retrying in {wait_time}s...")                    time.sleep(wait_time)                    continue                else:                    print(f"Error: Status code {response.status_code}")                    return None            soup = BeautifulSoup(response.text, 'html.parser')            # Extract race_id from URL            race_id_match = re.search(r'race_id=(\w+)', url)            if not race_id_match:                print(f"Warning: Could not extract race_id from {url}")                return None            generated_id = race_id_match.group(1)            # SKIP CHECK            if existing_race_ids and generated_id in existing_race_ids:                print(f"Skipping {generated_id} (Already exists)")                return None            # メタデータ抽出（netkeibaのNARページ構造）            date_text = ""            venue_text = ""            race_num_text = ""            race_name_text = ""            grade_text = ""            course_type = ""            distance = ""            rotation = ""            weather = ""            condition = ""            # タイトルから情報抽出            title = soup.title.text if soup.title else ""            # レース名            race_name_elem = soup.select_one(".RaceName")            if race_name_elem:                race_name_text = race_name_elem.text.strip()            # レースデータ01から抽出            rd1 = soup.select_one(".RaceData01")            if rd1:                txt = rd1.text.strip()                # 天候                if "天候:晴" in txt: weather = "晴"                elif "天候:曇" in txt: weather = "曇"                elif "天候:小雨" in txt: weather = "小雨"                elif "天候:雨" in txt: weather = "雨"                elif "天候:雪" in txt: weather = "雪"                # 馬場状態                if "馬場:良" in txt: condition = "良"                elif "馬場:稍" in txt: condition = "稍重"                elif "馬場:重" in txt: condition = "重"                elif "馬場:不良" in txt: condition = "不良"                # コース・距離                match = re.search(r'(芝|ダ|障)(\d+)m', txt)                if match:                    ctype_raw = match.group(1)                    if ctype_raw == "芝": course_type = "芝"                    elif ctype_raw == "ダ": course_type = "ダート"                    elif ctype_raw == "障": course_type = "障害"                    distance = match.group(2)                # 回り                if "右" in txt: rotation = "右"                elif "左" in txt: rotation = "左"                elif "直線" in txt: rotation = "直"            # 日付・会場（タイトルやその他から）            date_match = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)', title)            if date_match:                date_text = date_match.group(1)            # 会場名（NAR競馬場リスト）            nar_venues = ["帯広", "門別", "盛岡", "水沢", "浦和", "船橋", "大井", "川崎",                          "金沢", "笠松", "名古屋", "園田", "姫路", "高知", "佐賀"]            for v in nar_venues:                if v in title:                    venue_text = v                    break            # レース番号            race_num_match = re.search(r'(\d+)R', title)            if race_num_match:                race_num_text = race_num_match.group(0)            # グレード            if "JpnI" in str(soup) or "ＪｐｎⅠ" in str(soup): grade_text = "JpnI"            elif "JpnII" in str(soup) or "ＪｐｎⅡ" in str(soup): grade_text = "JpnII"            elif "JpnIII" in str(soup) or "ＪｐｎⅢ" in str(soup): grade_text = "JpnIII"            # テーブル抽出            target_table = soup.find("table", id="All_Result_Table")            if not target_table:                print(f"Warning: Result table not found in {url}")                return None            rows = target_table.find_all("tr", class_="HorseList")            data = []            for row in rows:                rank_elem = row.select_one(".Rank")                rank = rank_elem.text.strip() if rank_elem else ""                waku_elem = row.select_one(".Waku")                waku = waku_elem.text.strip() if waku_elem else ""                umaban_elem = row.select_one(".Num")                umaban = umaban_elem.text.strip() if umaban_elem else ""                horse_name_elem = row.select_one(".Horse_Name a")                horse_name = horse_name_elem.text.strip() if horse_name_elem else ""                horse_url = horse_name_elem.get("href") if horse_name_elem else ""                horse_id = ""                if horse_url:                    m = re.search(r'/horse/(\d+)', horse_url)                    if m:                        horse_id = m.group(1)                # その他のデータ                cells = row.find_all("td")                def get_cell_text(class_name):                    for cell in cells:                        if class_name in cell.get("class", []):                            return cell.text.strip()                    return ""                row_data = {                    '着順': rank,                    '枠': waku,                    '馬番': umaban,                    '馬名': horse_name,                    'horse_id': horse_id,                    '性齢': get_cell_text("Barei"),                    '斤量': get_cell_text("Weight"),                    '騎手': get_cell_text("Jockey"),                    'タイム': get_cell_text("Time"),                    '着差': get_cell_text("Diff"),                    '人気': get_cell_text("Ninki"),                    '単勝オッズ': get_cell_text("Odds"),                    '後3F': get_cell_text("Time_Last"),                    '厩舎': "",                    '馬体重(増減)': get_cell_text("Weight_Diff")                }                data.append(row_data)            df = pd.DataFrame(data)            # メタデータ追加            df['日付'] = date_text            df['会場'] = venue_text            df['レース番号'] = race_num_text            df['レース名'] = race_name_text            df['重賞'] = grade_text            df['距離'] = distance            df['コースタイプ'] = course_type            df['天候'] = weather            df['馬場状態'] = condition            df['回り'] = rotation            df['race_id'] = generated_id            # === 過去成績と血統の取得 ===            print(f"  Fetching past performance & pedigree for {len(df)} horses...")            for idx in range(len(df)):                horse_id = df.at[idx, 'horse_id']                # 過去成績の初期化                for n in range(1, 6):                    prefix = f"past_{n}_"                    for field in ['date', 'rank', 'time', 'run_style', 'race_name', 'last_3f',                                  'horse_weight', 'jockey', 'condition', 'odds', 'weather',                                  'distance', 'course_type']:                        df.at[idx, prefix + field] = ""                # 血統の初期化                df.at[idx, 'father'] = ""                df.at[idx, 'mother'] = ""                df.at[idx, 'bms'] = ""                if not horse_id or not str(horse_id).isdigit():                    continue                # 過去成績取得                try:                    past_df = scraper.get_past_races(horse_id, date_text, n_samples=5)                    if not past_df.empty:                        for n, (_, past_row) in enumerate(past_df.iterrows()):                            if n >= 5:                                break                            prefix = f"past_{n+1}_"                            df.at[idx, prefix + 'date'] = past_row.get('date', "")                            df.at[idx, prefix + 'rank'] = past_row.get('rank', "")                            df.at[idx, prefix + 'time'] = past_row.get('time', "")                            df.at[idx, prefix + 'run_style'] = past_row.get('run_style', "")                            df.at[idx, prefix + 'race_name'] = past_row.get('race_name', "")                            df.at[idx, prefix + 'last_3f'] = past_row.get('last_3f', "")                            df.at[idx, prefix + 'horse_weight'] = past_row.get('horse_weight', "")                            df.at[idx, prefix + 'jockey'] = past_row.get('jockey', "")                            df.at[idx, prefix + 'condition'] = past_row.get('condition', "")                            df.at[idx, prefix + 'odds'] = past_row.get('odds', "")                            df.at[idx, prefix + 'weather'] = past_row.get('weather', "")                            df.at[idx, prefix + 'distance'] = past_row.get('distance', "")                            df.at[idx, prefix + 'course_type'] = past_row.get('course_type', "")                except Exception as e:                    pass                # 血統取得                try:                    profile = scraper.get_horse_profile(horse_id)                    if profile:                        df.at[idx, 'father'] = profile.get('father', "")                        df.at[idx, 'mother'] = profile.get('mother', "")                        df.at[idx, 'bms'] = profile.get('bms', "")                except Exception as e:                    pass            # メモリ解放            gc.collect()            # 最終カラム整合性チェック - 82カラムを厳密に保証            df = df.reindex(columns=EXPECTED_COLUMNS, fill_value="")            print(f"✅ Scraped {len(df)} rows with full 82-column data.")            return df        except Exception as e:            if attempt < max_retries - 1:                wait_time = 2 ** attempt                print(f"Error: {e}, retrying in {wait_time}s...")                time.sleep(wait_time)            else:                print(f"❌ Failed after {max_retries} attempts: {e}")                return None    return Nonedef run_nar_scraping(year, start_month=1, end_month=12, save_dir='data/raw', existing_race_ids=None, save_callback=None):    """    NAR races scraping with memory efficiency and progress tracking.    """    start_date = date(int(year), int(start_month), 1)    import calendar    last_day = calendar.monthrange(int(year), int(end_month))[1]    end_date = date(int(year), int(end_month), last_day)    today = date.today()    if end_date > today:        end_date = today    print(f'=== NAR スクレイピング開始 ===')    print(f'期間: {start_date} ～ {end_date}')    print(f'完全な82カラムデータ（過去成績・血統含む）を取得します')    print(f'ランダムディレイ (1.0-2.0秒) でレート制限を回避')    curr = start_date    failed_races = []    total_processed = 0    total_days = (end_date - start_date).days + 1    with tqdm(total=total_days, desc="日付処理中") as pbar:        while curr <= end_date:            d_str = curr.strftime('%Y%m%d')            url = f'https://nar.netkeiba.com/top/race_list_sub.html?kaisai_date={d_str}'            try:                 time.sleep(random.uniform(0.5, 1.0))                 headers = {'User-Agent': 'Mozilla/5.0'}                 resp = requests.get(url, headers=headers, timeout=15)                 resp.encoding = 'EUC-JP'                 soup = BeautifulSoup(resp.text, 'html.parser')                 links = soup.select('a[href*="race/result.html"]')                 if links:                     print(f'\n📅 {curr}: {len(links)}件のレースを発見')                     for link in tqdm(links, desc=f"  {curr}", leave=False):                         href = link.get('href')                         if href.startswith('../'):                             full_url = f'https://nar.netkeiba.com/{href.replace("../", "")}'                         elif href.startswith('http'):                             full_url = href                         else:                             full_url = f'https://nar.netkeiba.com{href}'                         # 通信前の重複チェック                         race_id_match = re.search(r'race_id=(\w+)', full_url)                         if race_id_match:                             extracted_id = race_id_match.group(1)                             if existing_race_ids and extracted_id in existing_race_ids:                                 continue                         try:                             df = scrape_nar_race(full_url, existing_race_ids=existing_race_ids, max_retries=3)                             if df is not None and not df.empty:                                 if save_callback:                                     save_callback(df)                                 total_processed += 1                             else:                                 race_id = extracted_id if 'extracted_id' in locals() else full_url                                 failed_races.append(race_id)                             time.sleep(random.uniform(1.0, 2.0))                             if total_processed % 5 == 0 and total_processed > 0:                                 print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")                                 gc.collect()                         except Exception as e_race:                             print(f'  ❌ Error scraping race {full_url}: {e_race}')                             race_id = extracted_id if 'extracted_id' in locals() else full_url                             failed_races.append(race_id)            except Exception as e:                print(f'❌ Error on {curr}: {e}')            curr += timedelta(days=1)            pbar.update(1)    print(f"\n{'='*50}")    print(f"✅ スクレイピング完了")    print(f"総処理件数: {total_processed}件")    print(f"失敗件数: {len(failed_races)}件")    if failed_races:        print(f"\n⚠️ 失敗したレース:")        for race_id in failed_races[:10]:            print(f"  - {race_id}")

In [ ]:
# 実行ブロック（欠損チェック機能付き）import osimport pandas as pdif YEAR:    os.makedirs(SAVE_DIR, exist_ok=True)    save_path = os.path.join(SAVE_DIR, 'database_nar.csv')    # 安全な追記関数（94カラム厳密保証）    def safe_append_csv(df_chunk, path):        import pandas as pd        import os        if not os.path.exists(path):            # 新規作成 - 94カラムヘッダーを確実に書き込み            df_chunk = df_chunk.reindex(columns=EXPECTED_COLUMNS, fill_value="")            df_chunk.to_csv(path, index=False)        else:            try:                # 既存ヘッダー読み込み                existing_cols = pd.read_csv(path, nrows=0).columns.tolist()                                # カラム数チェック                if len(existing_cols) != 82:                    print(f"⚠️ 警告: 既存ファイルのカラム数が{len(existing_cols)}です（期待値: 82）")                    print(f"  ファイルを確認してください: {path}")                    return                                # カラム順序を既存ファイルに合わせる                df_aligned = df_chunk.reindex(columns=existing_cols, fill_value="")                                # 追記                df_aligned.to_csv(path, mode='a', header=False, index=False)            except Exception as e:                print(f"❌ 保存エラー: {e}")    # 既存データの読み込みと欠損チェック    existing_race_ids = set()    if os.path.exists(save_path):        print('既存データを読み込み中...')        try:            existing_df = pd.read_csv(save_path, low_memory=False)            if 'race_id' in existing_df.columns:                # race_idを文字列に変換                existing_df['race_id'] = existing_df['race_id'].astype(str).str.replace(r'\.0$', '', regex=True)                # 全94カラムのチェック（欠損がない行のみ「完全」）                complete_mask = pd.Series(True, index=existing_df.index)                                # 基本情報カラムチェック                basic_cols = ['日付', '会場', 'レース番号', 'レース名', 'コースタイプ', '距離',                               '天候', '馬場状態', '馬名', 'horse_id']                for col in basic_cols:                    if col in existing_df.columns:                        complete_mask = complete_mask & existing_df[col].notna() & \                                        (existing_df[col] != '') & (existing_df[col] != 'nan')                                # 過去成績カラムチェック（past_1のみチェック - あれば全体OKとみなす）                if 'past_1_date' in existing_df.columns:                    has_past_data = existing_df['past_1_date'].notna() & \                                    (existing_df['past_1_date'] != '') & \                                    (existing_df['past_1_date'] != 'nan')                    complete_mask = complete_mask & has_past_data                                # 血統カラムチェック                if 'father' in existing_df.columns:                    has_pedigree = existing_df['father'].notna() & \                                   (existing_df['father'] != '') & \                                   (existing_df['father'] != 'nan')                    complete_mask = complete_mask & has_pedigree                complete_races = existing_df[complete_mask]['race_id'].unique()                existing_race_ids = set(complete_races)                total_races = existing_df['race_id'].nunique()                complete_count = len(existing_race_ids)                incomplete_count = total_races - complete_count                                print(f'既存データ: {total_races}レース')                print(f'  完全データ: {complete_count}レース')                print(f'  不完全データ: {incomplete_count}レース（再取得対象）')                                # カラム数確認                actual_cols = len(existing_df.columns)                if actual_cols != 82:                    print(f'⚠️ 警告: 既存データのカラム数が{actual_cols}です（期待値: 82）')                            except Exception as e:            print(f'既存データの読み込みエラー（新規作成します）: {e}')    print(f'\n{YEAR}年のNARデータを取得します...')    print(f'保存先: {save_path}')        run_nar_scraping(        YEAR,        START_MONTH,        END_MONTH,        save_dir=SAVE_DIR,        existing_race_ids=existing_race_ids,        save_callback=lambda df: safe_append_csv(df, save_path)    )        # 最終確認    if os.path.exists(save_path):        final_df = pd.read_csv(save_path, nrows=0)        print(f'\n✅ 完了しました。')        print(f'最終カラム数: {len(final_df.columns)}')        if len(final_df.columns) == 82:            print('🎉 カラム数が正しいです（94カラム）')        else:            print(f'⚠️ カラム数が不正です（期待: 82, 実際: {len(final_df.columns)}）')else:    print('年度が設定されていません。')